# Finding Similar Songs with FastPath

This notebook uses the **FastPath** algorithm to turn a Guitar-Hero-style
song's note sequence into a vector embedding. Then it finds similar songs by comparing
those embeddings. We apply same technique used for customer-journey / sequence analysis
to Guitar Hero note charts instead.

**Graph model** (already loaded into the shared Aura instance before class — see
`data/` in the repo root if you're curious how, but you won't need to run anything
there today):

![Graph schema: a Song node connected via FIRST_NOTE and LAST_NOTE to a chain of Note nodes linked by NEXT_NOTE](https://raw.githubusercontent.com/smithna/clone-hero-fastpath/main/graph_schema.svg)

```
(:Song {song_id, title, artist, genre, duration, ...})
(:Note {timestamp, button})
(:Song)-[:FIRST_NOTE]->(:Note)-[:NEXT_NOTE]->(:Note)-> ...(:Note)<-[:LAST_NOTE]-(:Song)
```

`timestamp` is seconds from the start of the song. `button` is an integer color code:
0=green, 1=red, 2=yellow, 3=blue, 4=orange, 5=open.

## Working in groups

For this lab, it's recommend that one person in your group shares their screen and drives;
everyone else can follow along or run the same cells too. We all share a single AuraDB instance
for this lab, but each of you opens your **own** GDS Session against it — your own Aura API credentials
(`AURA_CLIENT_ID`) are what keep your session name unique from everyone else's, since
the AuraDB instance itself (`AURA_INSTANCEID`) is the same for the whole room.

You have **~25 minutes** — this is most people's first hands-on time with FastPath, so
don't worry about covering everything. Cells from **Connect** through **Look up a
song** are the core path. Get there first. Everything after that (explainability
plots, near-duplicate detection) is bonus material: self-contained, in any order, only
if your group has time left. 💬 marks a discussion prompt, 🔧 marks a config tweak
worth trying — pick whichever interest your group, nobody needs to hit all of them.

This lab runs in **Google Colab**. Run the **Setup** cell below first: it installs the
dependencies and clones this repo's assets into the runtime. Then store your credentials as
**Colab Secrets** (the key icon in the left sidebar) using these names: `NEO4J_URI`,
`NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`, `AURA_INSTANCEID`, `AURA_CLIENT_ID`,
`AURA_CLIENT_SECRET`, `AURA_PROJECT_ID`. Everyone shares the same
`NEO4J_URI`/`NEO4J_USERNAME`/`NEO4J_PASSWORD`/`AURA_INSTANCEID` (the shared AuraDB instance),
but each person needs their **own** `AURA_CLIENT_ID`/`AURA_CLIENT_SECRET`/`AURA_PROJECT_ID` to
create an Aura Graph Analytics session. (Prefer local Jupyter? The setup cell falls back to a
`.env` file following `.env.example`.)

## Setup (Google Colab)

Run this cell first. In **Google Colab** the runtime is ephemeral, so it installs the lab dependencies every session. On a **local Jupyter** kernel it detects your existing virtual environment and skips the install. This cell also clones the exercise Github repository into the Colab instance, althought is not needed. All the code in this repository will be available in the Files icon on the left panel, feel free to explore the code to get familiar data ETL.


In [ ]:
# --- Environment setup (Colab-first, local fallback) -------------------------
# Keep this package list in sync with requirements.txt (the source of truth).
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess

    # 1. Install the pinned lab dependencies into the ephemeral Colab runtime.
    print("Google Colab detected -- installing lab dependencies...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "neo4j>=5.28,<6",
         "python-dotenv>=1.0,<2",
         "graphdatascience==2.0a1",
         "matplotlib>=3.8,<4"],
        check=True,
    )
    print("Dependencies installed.")

    # 2. Clone the repo so supporting assets (e.g. graph_schema.svg) are present,
    #    then work from inside it. Safe to re-run: skips the clone if it exists.
    REPO_URL = "https://github.com/smithna/clone-hero-fastpath.git"
    REPO_BRANCH = "main"          # change to your feature branch if needed
    REPO_DIR = "/content/clone-hero-fastpath"
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} ({REPO_BRANCH})...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
             REPO_URL, REPO_DIR],
            check=True,
        )
    else:
        print("Repo already cloned.")
    os.chdir(REPO_DIR)
    print("Working directory:", os.getcwd())
else:
    print("Local kernel detected -- assuming requirements.txt is already installed in your venv.")

In [ ]:
import os
from collections import Counter
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from neo4j import GraphDatabase
from graphdatascience.session import (
    GdsSessions,
    AuraAPICredentials,
    AlgorithmCategory,
    DbmsConnectionInfo,
)

## Connect

In [ ]:
# --- Credentials (Colab-first, local fallback) ------------------------------
# Lookup order for each value:
#   1. Google Colab Secrets  (the key icon in the left sidebar) -- recommended.
#   2. Environment / local .env file (python-dotenv) on your own machine.
#   3. An interactive prompt if the value is still missing.
# Store these names as Colab Secrets:
#   NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE, AURA_INSTANCEID,
#   AURA_CLIENT_ID, AURA_CLIENT_SECRET, AURA_PROJECT_ID

if not IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv()

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_secret(name, required=True, default=None, secret=False):
    """Resolve a credential: Colab Secrets -> environment/.env -> prompt."""
    value = None
    if userdata is not None:                       # 1. Colab Secrets
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:                                  # 2. Environment / .env
        value = os.environ.get(name)
    if not value and required:                     # 3. Interactive prompt
        prompt = f"Enter {name}: "
        value = getpass(prompt) if secret else input(prompt)
    if not value:
        value = default
    if value is not None:                          # expose downstream via env
        os.environ[name] = value
    return value


NEO4J_URI      = get_secret("NEO4J_URI")
NEO4J_USERNAME = get_secret("NEO4J_USERNAME")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", secret=True)
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", required=False, default="neo4j")

AURA_CLIENT_ID     = get_secret("AURA_CLIENT_ID")
AURA_CLIENT_SECRET = get_secret("AURA_CLIENT_SECRET", secret=True)
AURA_PROJECT_ID    = get_secret("AURA_PROJECT_ID")
AURA_INSTANCE_ID   = get_secret("AURA_INSTANCEID")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j.")

sessions = GdsSessions(
    api_credentials=AuraAPICredentials(AURA_CLIENT_ID, AURA_CLIENT_SECRET, AURA_PROJECT_ID)
)

## Size and create a GDS Session

We estimate memory from the graph's *actual* size.


In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run("""
        MATCH (s:Song) WITH count(s) AS songs
        MATCH (n:Note) WITH songs, count(n) AS notes
        MATCH ()-[:FIRST_NOTE]->()
        WITH songs, notes, count(*) AS firstNotes
        MATCH ()-[:NEXT_NOTE]->() RETURN songs, notes, firstNotes, count(*) AS nextNotes
    """).single()

song_count = counts["songs"]
note_count = counts["notes"]
node_count = song_count + note_count
relationship_count = counts["firstNotes"] + counts["nextNotes"]

print(f"{node_count:,.0f} nodes ({song_count} songs, {note_count:,.0f} notes), {relationship_count:,.0f} relationships to project")

In [ ]:
memory_estimate = sessions.estimate(
    node_count=node_count,
    relationship_count=relationship_count,
    algorithm_categories=[AlgorithmCategory.SIMILARITY, AlgorithmCategory.NODE_EMBEDDING],
    node_label_count=2,       # Song, Note
    node_property_count=3,    # outputTime, timestamp, button
    relationship_property_count=0,
)
memory_estimate

In [ ]:
# Session names must be unique within the Aura project. We share one AuraDB instance
# for this lab, so AURA_INSTANCE_ID is the same for everyone -- AURA_CLIENT_ID is what
# actually is unique per participant, since each of us has our own Aura API credentials.
SESSION_NAME = f"similar-song-fastpath-{AURA_CLIENT_ID}"

gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=memory_estimate,
    db_connection=DbmsConnectionInfo(
        aura_instance_id=AURA_INSTANCE_ID,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD,
    ),
)

## Build the graph projection

We have loaded Aura with a graph in the shape FastPath wants: a `Song` connected to a chain of
`Note`s via `FIRST_NOTE`/`NEXT_NOTE`. There's also a `LAST_NOTE` relationship straight from the
`Song` to its final note. That last piece lets this query use a **quantified path
pattern** (QPP) to walk from the first note to the last, with both endpoints already known.

`(s)-[:FIRST_NOTE]->()-[:NEXT_NOTE]->*(l)`

The same can also use QPP syntax to
cap a walk at a fixed number of hops (`{0,20}`) or add a `WHERE` condition to stop based on a condition,
if you only wanted FastPath to see part of a journey instead of the
whole thing.

This is a **path** data model — each `Note` points to exactly one next `Note`.

💬 **Discuss**: what would change if notes were modeled **hub-and-spoke** instead —
say, `(:Song)-[:HAS_NOTE]->(:Note)`, with no `NEXT_NOTE` chain at all?
Could you still build a FastPath projection from that shape? What would the projection
query need to do differently, and what (if anything) would you gain or lose compared to
the path model above? *(Hint: FastPath needs to walk each song's events in time order —
where does that ordering come from in each model?)*


In [ ]:
def create_projection():
    query = """
    MATCH (s:Song)-[:LAST_NOTE]->(l)
    MATCH p=(s)-[:FIRST_NOTE]->()-[:NEXT_NOTE]->*(l)
    UNWIND relationships(p) AS r
    WITH startNode(r) AS source, endNode(r) AS target, r
    RETURN gds.graph.project.remote(
        source, target,
        {
            sourceNodeLabels: labels(source),
            targetNodeLabels: labels(target),
            sourceNodeProperties: source{.button, .timestamp},
            targetNodeProperties: target{.button, .timestamp},
            relationshipType: type(r)
        }
    )
    """
    g, _result = gds.graph.project(graph_name="songs", query=query)
    return g


g_songs = create_projection()
print(f'{g_songs.node_count():,.0f} nodes, {g_songs.relationship_count():,.0f} relationships')

## Compute FastPath embeddings

💬 **Discuss (before running the cell below)**: every parameter here was chosen
deliberately for song data. Before
reading our reasoning, guess — why might each of these make sense for a *song*
specifically?
- `time_node_property="timestamp"` is required, not just optional
- `decay_factor=0.0` (no recency bias)
- one shared `output_time` for every song, rather than a per-song value
- `dimension=128`
- `smoothing_window=2` and `smoothing_rate=10/MAX_ELAPSED_TIME` together

**Our reasoning**, once you've discussed:
- `time_node_property` — without it, FastPath falls back to treating notes as evenly
  spaced by position, throwing away real rhythm/timing.
- `output_time` shared across all songs aligns every song to the same starting line,
  so comparisons are beginning-to-beginning rather than ending-to-ending.
- `decay_factor=0.0` — a customer journey trends toward "what happens next," so recent
  events matter more; a song is a closed structure where the opening riff matters as
  much as the outro.
- `dimension=128` — the grid has `NUM_ELAPSED_TIMES x 6 buttons = 120` cells, each
  assigned an independent random vector. Random vectors in `d` dimensions have a
  typical pairwise cosine similarity of about `1/sqrt(d)`; 128 keeps the worst-case
  pair comfortably close to orthogonal.
- `smoothing_window` and `smoothing_rate` work as a pair — `smoothing_window` sets how
  many neighboring time buckets a note's timing can bleed into, `smoothing_rate` sets
  how fast that influence decays as you move away from the note's exact bucket. We
  picked a narrow window (2) with a sharp decay: just enough that a note landing right
  on a bucket boundary doesn't lose its signal entirely to one side, without smearing a
  song's rhythmic shape into mush. Note timestamps come straight from the chart file,
  so they're already precise — a customer journey, where event timing can be fuzzier,
  might reasonably want a gentler rate to compensate.

🔧 **Try it**: change `decay_factor` to `0.3`, or open up `smoothing_window`/
`smoothing_rate` to something gentler, then re-run this cell plus **Find similar songs**
and **Look up a song** below — do the matches for your song change? Does the shift
match what you'd predict from the reasoning above?


In [ ]:
# Single source of truth: must match fetch_charts.py's --max-minutes cap (currently 5
# min = 300s), since that's what guarantees every song's notes fit inside the window
# below. Change this if you ever refetch with a different duration cap.
MAX_SONG_DURATION = 300  # seconds

# output_time must be >= the longest possible song, or that song's last notes get
# excluded (FastPath drops events at or after output_time). +1s is the same small
# buffer we used for the old per-song outputTime, here applied once, globally.
OUTPUT_TIME = MAX_SONG_DURATION + 1

# max_elapsed_time must be > output_time, or a song's very first notes (elapsed_time =
# output_time - 0 = output_time) get excluded too. A few seconds of headroom is plenty.
MAX_ELAPSED_TIME = OUTPUT_TIME + 9

NUM_ELAPSED_TIMES = 20  # ~15.5s per grid bucket (MAX_ELAPSED_TIME / 20)

mutate_result = gds.fast_path.mutate(
    g_songs,
    mutate_property="fastPathEmbedding",
    base_node_label="Song",
    event_node_label="Note",
    first_relationship_type="FIRST_NOTE",
    next_relationship_type="NEXT_NOTE",
    time_node_property="timestamp",
    categorical_event_properties=["button"],
    output_time=OUTPUT_TIME,
    max_elapsed_time=MAX_ELAPSED_TIME,
    num_elapsed_times=NUM_ELAPSED_TIMES,
    smoothing_window=2,
    smoothing_rate=10 / MAX_ELAPSED_TIME,
    decay_factor=0.0,
    dimension=128,
    random_seed=42,   # fixed so everyone in the lab gets comparable results
)
mutate_result

## Find similar songs (KNN)

Cosine similarity over the FastPath embeddings, written back to the projection (not the Aura DB graph) as `SIMILAR_TO`
relationships. This sets up the KNN genre classifier further down.

In [ ]:
knn_result = gds.knn.mutate(
    g_songs,
    mutate_relationship_type="SIMILAR_TO",
    mutate_property="similarity",
    node_labels=["Song"],
    node_properties="fastPathEmbedding",
    top_k=10,
)
knn_result.relationships_written

In [ ]:
knn_df = gds.graph.relationships.stream(g_songs, ["SIMILAR_TO"], ["similarity"])
knn_df.set_index("sourceNodeId", inplace=True)
knn_df.sort_index(inplace=True)

## Rerank with Euclidean distance

Cosine similarity ignores vector magnitude, and FastPath's magnitude tracks note
*density/intensity* (more/denser notes -> larger vector sum). Two songs can look
identical on cosine alone despite one being much busier than the other. Filtering by
cosine first (same "shape" of activity) then reranking by Euclidean distance (same
"intensity") gives a better final ordering.

🔧 **Try it**: `rerank_similarity()` below blends `similarity_cosine * 0.6 +
similarity_euclidean * 0.4`. Try shifting that weight (e.g. 0.9/0.1 or 0.3/0.7) and
re-run **Look up a song** — how much does the ranking move?


In [ ]:
def rerank_similarity(source_node_id, top_n=5):
    cosine_df = knn_df.loc[[source_node_id]]
    targets = cosine_df["targetNodeId"].to_list()
    euclidean_df = gds.knn.filtered.stream(
        g_songs,
        node_labels=["Song"],
        node_properties={"fastPathEmbedding": "EUCLIDEAN"},
        source_node_filter=source_node_id,
        target_node_filter=targets,
        top_k=10,
        seed_target_nodes=True,
    )
    merged = cosine_df.merge(
        euclidean_df, how="outer", left_on="targetNodeId", right_on="node2",
        suffixes=("_cosine", "_euclidean"),
    )
    merged.drop(columns=["relationshipType", "node1", "node2"], inplace=True)
    merged["similarity_final"] = merged["similarity_cosine"] * 0.6 + merged["similarity_euclidean"] * 0.4
    merged.sort_values("similarity_final", ascending=False, inplace=True)
    return merged.head(top_n)

## Look up a song and see what's similar

Pick a song by (partial) title, see its nearest neighbors by name and
artist instead of raw node ids.

🔧 **Try it**: search for your own favorite, or pick from these to compare across
genres —
- `"castle of glass"` (Linkin Park)
- `"7 rings"` (Ariana Grande)
- `"billie jean"` (Michael Jackson)
- `"back in black"` (AC/DC — there's also a joke chart, `"but the band is on crack"`,
  worth comparing against the original)
- `"viva la vida"` (Coldplay)

💬 **Discuss**: FastPath only sees note timing and button color — no genre, no tempo
label, no lyrics. Do the top matches make musical sense to you? Where does "shape
similarity" line up with genre, and where does it diverge?


In [ ]:
def find_song(title_contains):
    return gds.run_cypher("""
        MATCH (s:Song) WHERE toLower(s.title) CONTAINS toLower($q)
        RETURN id(s) AS nodeId, s.title AS title, s.artist AS artist, s.song_id AS song_id
        ORDER BY s.title
    """, {"q": title_contains})


def similar_songs(node_id, top_n=5):
    ranked = rerank_similarity(node_id, top_n=top_n)
    ids = ranked["targetNodeId"].tolist()
    songs_df = gds.run_cypher("""
        UNWIND $ids AS nodeId
        MATCH (s:Song) WHERE id(s) = nodeId
        RETURN id(s) AS nodeId, s.title AS title, s.artist AS artist, s.genre AS genre
    """, {"ids": ids})
    return ranked.merge(songs_df, left_on="targetNodeId", right_on="nodeId")[
        ["nodeId", "title", "artist", "genre", "similarity_cosine", "similarity_euclidean", "similarity_final"]
    ]

In [ ]:
# Try any song from the shared dataset -- a partial, case-insensitive title match
matches = find_song("shake it off")
matches

In [ ]:
song_node_id = int(matches.iloc[0]["nodeId"])
top_matches = similar_songs(song_node_id)
top_matches

## Bonus: compare a song's best and worst match directly

*Optional — only if your group has time. Skip to Cleanup otherwise.*

Is a 0.93 similarity score actually good? Compared to what? This section plots the
lookup song against both its single best match and its single worst match (out of all
300 songs), using the same note-density grid FastPath is built on (time bucket ×
button color) — the "Reference Grid" from the FastPath deck.

Two plots, not one: a percentage heatmap (each song scaled to its own total, so its
pattern stays readable) plus a separate bar chart of absolute note counts, since a
shared color scale would wash out the sparser song.

💬 **Discuss (before running)**: guess first — will the worst match look nothing like
the lookup song, or could two very different songs still score awkwardly similar? What
would a surprising result tell you about where this algorithm's blind spots are?


In [ ]:
BUTTON_NAMES = ["Green", "Red", "Yellow", "Blue", "Orange", "Open"]
BUCKET_WIDTH = MAX_ELAPSED_TIME / NUM_ELAPSED_TIMES

def get_song_notes(node_id):
    return gds.run_cypher("""
        MATCH (s:Song) WHERE id(s) = $nodeId
        MATCH (s)-[:FIRST_NOTE]->(first:Note)
        MATCH (first)-[:NEXT_NOTE*0..]->(n:Note)
        RETURN n.timestamp AS timestamp, n.button AS button
        ORDER BY n.timestamp
    """, {"nodeId": node_id})


def note_density_grid(notes_df):
    grid = np.zeros((len(BUTTON_NAMES), NUM_ELAPSED_TIMES), dtype=int)
    for _, row in notes_df.iterrows():
        bucket = min(int(row["timestamp"] // BUCKET_WIDTH), NUM_ELAPSED_TIMES - 1)
        grid[int(row["button"]), bucket] += 1
    return grid


# Full pairwise cosine similarity across all 300 songs -- needed to find the single
# LEAST similar song, since KNN only ever kept each song's top 10 nearest matches.
embedding_lookup = gds.graph.node_properties.stream(g_songs, ["fastPathEmbedding"], node_labels=["Song"]).set_index("nodeId")
embedding_matrix = np.vstack(embedding_lookup["fastPathEmbedding"].values)
normalized_embeddings = embedding_matrix / np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
full_similarity = normalized_embeddings @ normalized_embeddings.T

# Full pairwise EUCLIDEAN similarity too (GDS's convention: 1/(1+distance)), used by
# duplicate detection below -- Euclidean is sensitive to embedding magnitude (which
# tracks note count), and that turns out to separate true duplicates from same-artist
# different-song pairs far more cleanly than cosine does.
squared_norms = np.sum(embedding_matrix ** 2, axis=1)
squared_distances = np.maximum(
    squared_norms[:, None] + squared_norms[None, :] - 2 * (embedding_matrix @ embedding_matrix.T), 0
)
full_euclidean_similarity = 1 / (1 + np.sqrt(squared_distances))

node_ids_ordered = embedding_lookup.index.to_list()
node_index = {nid: i for i, nid in enumerate(node_ids_ordered)}


def worst_match(node_id):
    i = node_index[node_id]
    sims = full_similarity[i].copy()
    sims[i] = np.nan  # exclude self
    worst_i = int(np.nanargmin(sims))
    return node_ids_ordered[worst_i], float(sims[worst_i])


def song_title(node_id):
    return gds.run_cypher(
        "MATCH (s:Song) WHERE id(s) = $nodeId RETURN s.title AS title, s.artist AS artist",
        {"nodeId": node_id},
    ).iloc[0]

In [ ]:
def compare_best_and_worst(node_id):
    # Best match: the exact same cosine + Euclidean reranking as "Look up a song".
    best_row = rerank_similarity(node_id, top_n=1).iloc[0]
    best_id = int(best_row["targetNodeId"])
    best_sim = best_row["similarity_final"]

    # Worst match: full pairwise cosine search finds the candidate (KNN can't), then
    # rerank that one candidate through Euclidean too, for a comparable final score.
    worst_id, worst_cosine = worst_match(node_id)
    worst_euclidean_df = gds.knn.filtered.stream(
        g_songs,
        node_labels=["Song"],
        node_properties={"fastPathEmbedding": "EUCLIDEAN"},
        source_node_filter=node_id,
        target_node_filter=[worst_id],
        top_k=1,
        seed_target_nodes=True,
    )
    worst_euclidean = float(worst_euclidean_df.iloc[0]["similarity"])
    worst_sim = worst_cosine * 0.6 + worst_euclidean * 0.4

    lookup, best, worst = song_title(node_id), song_title(best_id), song_title(worst_id)

    grids = {
        "lookup": note_density_grid(get_song_notes(node_id)),
        "best": note_density_grid(get_song_notes(best_id)),
        "worst": note_density_grid(get_song_notes(worst_id)),
    }
    note_counts = {key: int(grid.sum()) for key, grid in grids.items()}
    # Percentage of each song's OWN total, independently scaled per panel -- this is
    # the "where is the activity" plot. It deliberately can't show density differences
    # between songs; that's what the bar chart below is for.
    percent_grids = {key: grid / grid.sum() * 100 for key, grid in grids.items()}

    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True)
    im = None
    row_specs = [
        (axes[0], "best", f"{best['artist']} - {best['title']} (BEST, sim={best_sim:.3f})"),
        (axes[1], "worst", f"{worst['artist']} - {worst['title']} (WORST, sim={worst_sim:.3f})"),
    ]
    for row_axes, key, other_title in row_specs:
        for ax, grid_key, title in zip(row_axes, ["lookup", key],
                                        [f"{lookup['artist']} - {lookup['title']}", other_title]):
            im = ax.imshow(percent_grids[grid_key], aspect="auto", cmap="viridis")
            ax.set_yticks(range(len(BUTTON_NAMES)))
            ax.set_yticklabels(BUTTON_NAMES)
            ax.set_title(title, fontsize=9)
    for ax in axes[1]:
        ax.set_xlabel(f"Time bucket (~{BUCKET_WIDTH:.0f}s each)")
    fig.colorbar(im, ax=axes, label="% of song's own notes")
    fig.suptitle(f"{lookup['artist']} - {lookup['title']}: note SHAPE -- best match (top row) vs worst match (bottom row)")
    plt.show()

    # Absolute note counts, shown separately since the heatmap above is deliberately
    # normalized per-song and can't convey density differences between songs.
    fig2, ax2 = plt.subplots(figsize=(6, 2.5))
    labels = [f"{lookup['artist']} - {lookup['title']}",
              f"{best['artist']} - {best['title']} (BEST)",
              f"{worst['artist']} - {worst['title']} (WORST)"]
    counts = [note_counts["lookup"], note_counts["best"], note_counts["worst"]]
    ax2.barh(labels, counts, color=["tab:gray", "tab:green", "tab:red"])
    ax2.set_xlabel("Total notes")
    ax2.set_title("Note density (absolute count)")
    plt.tight_layout()
    plt.show()

    return best_sim, worst_sim


compare_best_and_worst(song_node_id)

## Bonus: find near-duplicate charts

*Optional — only if your group has time.*

The single highest-similarity pair in the whole dataset (0.9998 cosine) turned out to
be the same U2 song charted twice — live and studio versions with slightly different
titles. Embedding similarity can double as **near-duplicate detection**, the same way
you'd deduplicate near-identical customer records or support tickets.

We use **Euclidean** here, not cosine — cosine ignores magnitude, and some artists have
multiple *different* songs charted in a consistently dense style (look for
`[Overchart]`) that score just as high on cosine (0.94-0.97) as true duplicates,
despite very different note counts. Euclidean separates these cleanly (true duplicates:
0.014-0.086; same-artist-different-song: 0.0003-0.0005) — but not with a hard
guarantee, so we back it with same-artist + fuzzy title matching too.

💬 **Discuss**: why three signals (Euclidean + same artist + fuzzy title) instead of
one strict threshold? Can you think of a *non-music* dataset — support tickets, log
sequences, customer journeys — where "several weak signals" beats "one strong
threshold" the same way?


In [ ]:
import itertools
import re
import difflib

def normalize_title(title):
    title = re.sub(r"<[^>]*>", "", title)      # strip html-ish tags like <color=...>
    title = re.sub(r"\([^)]*\)", "", title)    # strip (...) qualifiers
    title = re.sub(r"\[[^\]]*\]", "", title)   # strip [...] qualifiers
    return re.sub(r"\s+", " ", title).strip().lower()


def title_similarity(a, b):
    return difflib.SequenceMatcher(None, normalize_title(a), normalize_title(b)).ratio()


EUCLIDEAN_THRESHOLD = 0.01
TITLE_THRESHOLD = 0.9

song_lookup = gds.run_cypher(
    "MATCH (s:Song) RETURN id(s) AS nodeId, s.title AS title, s.artist AS artist"
).set_index("nodeId")

# Scan every pair directly from the full Euclidean matrix, rather than relying on
# knn_df's top-10-per-song (cosine-derived) candidate set -- a true duplicate should
# always be extremely close in both metrics, but there's no reason to depend on that.
pairs = []
n = len(node_ids_ordered)
for a, b in itertools.combinations(range(n), 2):
    sim = float(full_euclidean_similarity[a, b])
    if sim < EUCLIDEAN_THRESHOLD:
        continue
    id_a, id_b = node_ids_ordered[a], node_ids_ordered[b]
    artist_a, artist_b = song_lookup.loc[id_a, "artist"], song_lookup.loc[id_b, "artist"]
    if (artist_a or "").strip().lower() != (artist_b or "").strip().lower():
        continue
    title_a, title_b = song_lookup.loc[id_a, "title"], song_lookup.loc[id_b, "title"]
    t_sim = title_similarity(title_a, title_b)
    if t_sim < TITLE_THRESHOLD:
        continue
    pairs.append({
        "sourceNodeId": id_a, "targetNodeId": id_b,
        "sourceArtist": artist_a, "sourceTitle": title_a, "targetTitle": title_b,
        "similarity": sim, "titleSimilarity": t_sim,
    })

likely_duplicates = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
likely_duplicates[["sourceArtist", "sourceTitle", "targetTitle", "similarity", "titleSimilarity"]]

Label the detected pairs by writing them back to the actual Neo4j database as a real,
persisted relationship -- not just an in-memory result. This is different from
`SIMILAR_TO`: `knn.mutate(...)` only ever wrote into the *in-memory GDS session graph*
(that's why the genre-classifier step earlier had to reuse `knn_df` instead of querying
`SIMILAR_TO` from the database). Here we deliberately write through `gds.run_cypher(...)`
so the labels persist after the session is gone and anyone can query them later.

Since everyone is working in the same Aura database, other classmates might have
already written `:LIKELY_DUPLICATE_OF` relationships to the database before you get here.
Feel free to add a property value to make your relationship unique.

In [ ]:
write_result = gds.run_cypher("""
    UNWIND $pairs AS pair
    MATCH (a:Song) WHERE id(a) = pair.source
    MATCH (b:Song) WHERE id(b) = pair.target
    MERGE (a)-[r:LIKELY_DUPLICATE_OF]->(b)
    SET r.embeddingSimilarity = pair.embeddingSimilarity, r.titleSimilarity = pair.titleSimilarity
""", {
    "pairs": likely_duplicates.rename(
        columns={"sourceNodeId": "source", "targetNodeId": "target", "similarity": "embeddingSimilarity"}
    )[["source", "target", "embeddingSimilarity", "titleSimilarity"]].to_dict("records")
})

print(f"Labeled {len(likely_duplicates)} likely-duplicate pairs as (:Song)-[:LIKELY_DUPLICATE_OF]->(:Song)")

## Cleanup

GDS Sessions are billed while running -- drop the projected graph and delete the
session when you're done.


In [ ]:
g_songs.drop()
gds.delete()
driver.close()